In [1]:
!nvidia-smi
import torch

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB"
    )

Fri Jun 12 04:19:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+
|   1  Tesla T4                       Off |   00000000:00:05.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        

## Step 1: Clone Repository and Setup Environment

In [2]:
import os
import getpass
from pathlib import Path

# Configuration
GITHUB_USER = "sattary"
REPO_NAME = "ali_proj"
BRANCH = "fix-review"  # Change if using different branch
PROJECT_DIR = "ali_proj"

print("Enter your GitHub Personal Access Token (PAT):")
PAT = getpass.getpass()
REPO_URL = f"https://{PAT}@github.com/{GITHUB_USER}/{REPO_NAME}.git"

# 1. Clone Repository
if not Path(PROJECT_DIR).exists():
    print(f"Cloning {REPO_NAME} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
else:
    print("Repository already cloned. Pulling latest changes...")
    !cd {PROJECT_DIR} && git pull origin {BRANCH}

%cd {PROJECT_DIR}

# 2. Install uv
print("\nInstalling uv...")
!pip install -q uv

# 3. Set MPLBACKEND for Kaggle compatibility
os.environ['MPLBACKEND'] = 'Agg'
print("\nSet MPLBACKEND=Agg for headless environments")

# 4. Sync Dependencies
print("\nSyncing dependencies...")
!uv sync

print("\n✓ Setup complete!")

Enter your GitHub Personal Access Token (PAT):
Cloning ali_proj (branch: fix-review)...
Cloning into 'ali_proj'...
remote: Enumerating objects: 1209, done.
remote: Counting objects: 100% (518/518), done.
remote: Compressing objects: 100% (286/286), done.
remote: Total 1209 (delta 308), reused 414 (delta 215), pack-reused 691 (from 1)
Receiving objects: 100% (1209/1209), 39.74 MiB | 41.27 MiB/s, done.
Resolving deltas: 100% (693/693), done.
/kaggle/working/ali_proj

Installing uv...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 69.4 MB/s eta 0:00:00:00:0100:01

Set MPLBACKEND=Agg for headless environments

Syncing dependencies...
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 76 packages in 1ms
Prepared 74 packages in 1m 13s                                           
░░░░░░░░░░░░░░░░░░░░ [0/74] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead 

In [3]:
!uv pip install ipywidgets

Using Python 3.12.13 environment at: /usr
Checked 1 package in 210ms


In [4]:
!git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 940 bytes | 313.00 KiB/s, done.
From https://github.com/sattary/ali_proj
   cb8776b..3115cf7  fix-review -> origin/fix-review
Updating cb8776b..3115cf7
Fast-forward
 src/phase_unwrap/training/train.py | 15 ++++++++++++++-
 1 file changed, 14 insertions(+), 1 deletion(-)


In [6]:
!ls

configs  notebooks	       README.md  src		     tests
docs	 physics_breakdown.md  results	  status_handoff.md  uv.lock
main.py  pyproject.toml        scripts	  test.bak


In [7]:
# Verify Git Repository
from pathlib import Path
import subprocess

# Check if we're in a git repo
result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True
)
if result.returncode == 0:
    repo_root = result.stdout.strip()
    print(f"✓ Git repository found at: {repo_root}")
    print(f"✓ Current directory: {Path.cwd()}")
else:
    print("Error: Not in a git repository!")
    print(f"Current directory: {Path.cwd()}")
    raise RuntimeError("Git repository not found")

# Show repo status
!git status

✓ Git repository found at: /kaggle/working/ali_proj
✓ Current directory: /kaggle/working/ali_proj
On branch fix-review
Your branch is up to date with 'origin/fix-review'.

nothing to commit, working tree clean


## Phase 1: Deterministic Generation (`generate`)

Generate the synthetic interferogram dataset to HDF5 shards. The cryptographic seed guarantees mathematically invariant noise topologies.

In [15]:
!uv sync

Resolved 97 packages in 1ms
Prepared 22 packages in 1.08s                                            
Uninstalled 1 package in 0.56ms
░░░░░░░░░░░░░░░░░░░░ [0/22] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 22 packages in 688ms                              
 ~ ali-proj==0.1.0 (from file:///kaggle/working/ali_proj)
 + asttokens==3.0.1
 + comm==0.2.3
 + decorator==5.3.1
 + executing==2.2.1
 + ipython==9.14.1
 + ipython-pygments-lexers==1.1.1
 + ipywidgets==8.1.8
 + jedi==0.20.0
 + jupyterlab-widgets==3.0.16
 + matplotlib-inline==0.2.2
 + parso==0.8.7
 + pexpect==4.9.0
 + prompt-toolkit==3.0.52
 + psutil==7.2.2
 + ptyprocess==0.7.0
 + pure-eval==0.2.3


In [13]:
!ls data

In [16]:
# Phase 1: Generate Data
!uv run phase-unwrap generate \
    --num-samples 10000 \
    --shard-size 100 \
    --out-dir data/kaggle_full \
    --seed 1337

print("\n✓ Data generation complete!")

Shards: 100%|█████████████████████████████████| 100/100 [00:50<00:00,  1.99it/s]
Generated 10000 samples across 100 shards in data/kaggle_full
Saved data config: data/kaggle_full/data_config.yaml

✓ Data generation complete!


## Phase 2: Hyperparameter Optimization (`tune`)

Use Optuna's Bayesian TPE algorithm to isolate the absolute lowest-error configuration. The best configuration is automatically saved to `runs/optuna/best_config.yaml`.

In [17]:
# Phase 2: Optuna Tune
# Automatically parallelizes across available GPUs
!uv run phase-unwrap tune \
    --use-amp \
    --n-trials 15 \
    --tune-epochs 20 \
    --study-name kaggle_10k_hpo \
    --n-workers 2 \
    --batch-size 64 \
    --data-dir data/kaggle_full/

print("\n✓ Tuning complete!")
print("Best config frozen to: runs/optuna/best_config.yaml")

Auto-detected 2 GPUs, mapping 2 workers: [0, 1]
[I 2026-06-12 04:29:02,646] A new study created in RDB with name: kaggle_10k_hpo
Running 15 trials (0 existing, 15 target)
Parallel HPO: 2 workers on GPUs [0, 1]
Trials per worker: ~7
[Worker 0 on GPU 0] Starting...
[Worker 1 on GPU 1] Starting...
  Trial 0 | epoch 1/20 | val MAE=4.6628 (best=4.6628)                           
Trial 0 Ep 2/20:   0%|                                  | 0/125 [00:00<?, ?it/s]  Trial 1 | epoch 1/20 | val MAE=4.6343 (best=4.6343)
  Trial 1 | epoch 2/20 | val MAE=4.7179 (best=4.6343)                           
Trial 1 Ep 3/20:   0%|                                  | 0/125 [00:00<?, ?it/s]  Trial 0 | epoch 2/20 | val MAE=4.7111 (best=4.6628)
  Trial 0 | epoch 3/20 | val MAE=4.7145 (best=4.6628)                           
Trial 0 Ep 4/20:   0%|                                  | 0/125 [00:00<?, ?it/s]  Trial 1 | epoch 3/20 | val MAE=4.6932 (best=4.6343)
  Trial 0 | epoch 4/20 | val MAE=4.7524 (best=4.6628)      

## Phase 3: Primary Training and Evaluation (`train`)

Train the baseline network to convergence using the frozen `best_config.yaml`.

In [31]:
!git pull

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 441 bytes | 220.00 KiB/s, done.
From https://github.com/sattary/ali_proj
   9dd8dcf..1cb1a6b  fix-review -> origin/fix-review
Updating 9dd8dcf..1cb1a6b
Fast-forward
 src/phase_unwrap/cli.py | 1 -
 1 file changed, 1 deletion(-)


In [ ]:
# Phase 3: Train Primary Network
import os

config_arg = (
    "--config runs/optuna/best_config.yaml"
    if os.path.exists("runs/optuna/best_config.yaml")
    else ""
)

!uv run phase-unwrap train \
    --use-amp \
    {config_arg} \
    --run-name exp_primary \
    --data-dir data/kaggle_full/ \
    --batch-size 32 \
    --epochs 100

print("\n✓ Primary training complete!")

[startup] device=cuda | amp=True | batch=32 | workers=4
[run_dir] runs/exp_primary
[data] dir=data/kaggle_full/ pattern=*.h5 | train=8000 val=1000 test=1000
[startup] torch.compile enabled
[noise] epoch=1 level=0.000
Epoch 1/100:   0%|                                     | 0/250 [00:00<?, ?it/s]W0612 07:15:43.702000 23556 torch/_inductor/utils.py:1717] [0/0] Not enough SMs to use max_autotune_gemm mode
/kaggle/working/ali_proj/.venv/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(
/kaggle/working/ali_proj/.venv/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _

## Phase 4: Statistical Validation (`multiseed`)

Defend against 'lucky seed' anomalies. Spawns completely independent training convergences using the same frozen configuration, and automatically aggregates the metrics into Mean ± Std.

In [ ]:
# Phase 4: Statistical Validation
import os
config_arg = "--config runs/optuna/best_config.yaml" if os.path.exists("runs/optuna/best_config.yaml") else ""

!uv run phase-unwrap multiseed \
    --use-amp \
    {config_arg} \
    --run-name exp_multiseed \
    --num-seeds 3 \
    --data-dir data/kaggle_full/

print("\n✓ Multiseed validation complete!")

## Phase 5: Architectural Ablation (`ablation`)

Mathematically prove the necessity of your custom topology by systematically crippling the network. Exports a rigorous LaTeX comparison table.

In [ ]:
# Phase 5: Architectural Ablation
import os
config_arg = "--config runs/optuna/best_config.yaml" if os.path.exists("runs/optuna/best_config.yaml") else ""

!uv run phase-unwrap ablation \
    --use-amp \
    {config_arg} \
    --out-table results/tables/ablation.tex \
    --data-dir data/kaggle_full/

print("\n✓ Ablation study complete!")

## Step 6: Zip and Download Results

Run this cell to zip the `runs/` and `results/` directories so you can render them on your local machine.

In [ ]:
import shutil
from IPython.display import FileLink

print("Zipping runs and results...")
shutil.make_archive('training_results', 'zip', 'runs/')
shutil.make_archive('tables_results', 'zip', 'results/')
print("✓ Done!")
display(FileLink('training_results.zip'))
display(FileLink('tables_results.zip'))